# 05 Multimodal Inputs

So far, we have been providing _only_ text inputs to our Agents. In this workbook we'll illustrate how we can provide multi-modal inputs, such as images and audio to our agents. That should be fun, considering that in the next workbook we'll be building a fully functional _chef_ agent.

LLMs such as GPT models from OpenAI, Claude models from Anthropic and Gemini models from Google can ingest text or image or audio inputs and generate text or image or audio and even video outputs.

In this workbook we'll show you how to provide image and audio inputs to our Agents, which we'll be encoding into Base64 format - which encodes a binary (base 2) to 64 bits. This enables us to efficiently represent and transmit binary data, such as images and audio, on text-based communication channels.

In [9]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    system_prompt="You are a helpful assistant",
)

First let's see how we can _feed_ an image to our agent. We'll upload an image

In [3]:
from ipywidgets import FileUpload
from IPython.display import display

# upload single (multiple=False) PNG (accept=".png") files only!
uploader = FileUpload(accept=".png", multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [5]:
print(uploader.value[0])

{'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x000001F22978D0C0>, 'last_modified': datetime.datetime(2026, 4, 14, 14, 8, 35, 140000, tzinfo=datetime.timezone.utc)}


In [6]:
# let's encode the image using base64 encoding
import base64

uploaded_file = uploader.value[0]
encoded_image = base64.b64encode(bytes(uploaded_file["content"])).decode("utf-8")

In [10]:
# now let's pass a multi-modal question to our agent and get a response
# we'll ask agent to describe the image.
from langchain_core.messages import HumanMessage

multimodal_question = HumanMessage(
    content=[
        {"type": "text", "text": "Give me a short description of the image."},
        {"type": "image", "base64": encoded_image, "mime_type": "image/png"},
    ]
)

response = agent.invoke({"messages": [multimodal_question]})
print(response["messages"][-1].content)

An alien desert stretchs toward a distant, gleaming capital with towering spires. A colossal blue planet dominates the left side of the sky, its pale light washing over jagged rock formations. In the foreground a blue-lit pool mirrors the city’s glow as the skyline rises through a haze of turquoise night.


## Audio Inputs
Next, let's see how to give an agent audio inputs - we'll be recording your input, encoding it and then passing it to the agent to process.

**NOTE:**
1. You will have to install sounddevice and the tqdm packages for this section to work - run `uv add sounddevice` and `uv add tqdm` from the command line after activating the environment.

In [5]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

In [ ]:
duration = 5  # seconds
sample_rate = 44100  # Hz

print("Recording....")
# record mono audio (channels=1)
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
for _ in tqdm(range(duration * 10)):
    time.sleep(0.1)
sd.wait()
print("Done!")

# save the audio to a bytes buffer
buffer = io.BytesIO()
write(buffer, sample_rate, audio)
wav_bytes = buffer.getvalue()
audio_base64 = base64.b64encode(wav_bytes).decode("utf-8")

The previous snippet records a 5 second audio only - stops recording if you continue to ramble after 5 seconds!

Here is an example of how you could record continuously until the user utters a stop word - we'll use "Go!" as the stop word.

**NOTE:** 
To use this snippet you'll have to instal the `SpeechRecognition` package (`uv add SpeechRecognition`) after activating the current environment.

```python
    import io
    import base64
    import queue
    import threading
    import numpy as np
    import sounddevice as sd
    from scipy.io.wavfile import write
    import speech_recognition as sr 

    sample_rate = 44100

    # Shared state
    audio_chunks = []
    stop_event = threading.Event()

    def record_audio():
        """Continuously records audio chunks into audio_chunks list."""
        def callback(indata, frames, time, status):
            if not stop_event.is_set():
                audio_chunks.append(indata.copy())

        with sd.InputStream(samplerate=sample_rate, channels=1, callback=callback):
            print("Recording... say 'Go!' to stop.")
            stop_event.wait()  # blocks until stop_event is set

    def listen_for_trigger():
        """Listens for the trigger word 'Go' and sets the stop event."""
        recognizer = sr.Recognizer()
        mic = sr.Microphone()

        with mic as source:
            recognizer.adjust_for_ambient_noise(source, duration=1)

        while not stop_event.is_set():
            with mic as source:
                try:
                    audio = recognizer.listen(source, timeout=3, phrase_time_limit=3)
                    text = recognizer.recognize_google(audio).lower()
                    print(f"Heard: {text}")
                    if "go" in text:
                        print("Trigger word detected! Stopping recording.")
                        stop_event.set()
                except (sr.WaitTimeoutError, sr.UnknownValueError):
                    pass  # no speech or unintelligible — keep listening
                except sr.RequestError as e:
                    print(f"Speech recognition error: {e}")
                    break

    # Run both threads concurrently
    record_thread = threading.Thread(target=record_audio)
    trigger_thread = threading.Thread(target=listen_for_trigger)

    record_thread.start()
    trigger_thread.start()

    record_thread.join()
    trigger_thread.join()

    # Combine all recorded chunks
    audio = np.concatenate(audio_chunks, axis=0)

    # Save to bytes buffer → base64 (same as before)
    buffer = io.BytesIO()
    write(buffer, sample_rate, audio)
    wav_bytes = buffer.getvalue()
    audio_base64 = base64.b64encode(wav_bytes).decode("utf-8")

    print(f"Captured {len(audio) / sample_rate:.1f} seconds of audio.")
```

In [ ]:
multimodal_question = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "First transcribe the audio to text and then do what the audio asks for.",
        },
        {"type": "audio", "base64": audio_base64, "mime_type": "audio/wav"},
    ]
)

response = agent.invoke({"messages": [multimodal_question]})
print(response["messages"][-1].content)